# 11.4 — Dynamic Programming

Dynamic programming (DP) plans in a known reinforcement-learning world by sweeping exact Bellman backups through value tables until values and policies agree with long-term consequence. In this lesson, you will build discounted returns, policy evaluation, policy improvement, policy iteration, and value iteration from scratch in NumPy on a tiny gridworld.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build dynamic programming one idea at a time. Run each cell in order and read the printed intermediate values — every value, action backup, policy update, and stopping rule is inspectable. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, transition tensors, Bellman backups, and assertions.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any tie-breaking demos.

### 1. A known gridworld model

Dynamic programming assumes we know the model: from every state and action, we know the next-state probabilities and rewards. We use a deterministic 3×3 grid with a terminal goal in the bottom-right. Each non-terminal move costs −1, and boundary moves leave the agent in the same cell.

In [ ]:
H_w, W_w = 3, 3
A_w = 4
actions_w = ["↑", "→", "↓", "←"]
deltas_w = [(-1, 0), (0, 1), (1, 0), (0, -1)]
goal_w = (2, 2)
def idx_w(r, c):
    return r * W_w + c
def rc_w(s):
    return divmod(s, W_w)
print("states:", H_w * W_w, "actions:", A_w, "goal:", idx_w(*goal_w))

▶ What you'll see: 9 states, 4 actions, and state 8 as the terminal goal.

In [ ]:
P_w = np.zeros((H_w * W_w, A_w, H_w * W_w))
R_w = np.zeros((H_w * W_w, A_w, H_w * W_w))
for s_w in range(H_w * W_w):
    r_w, c_w = rc_w(s_w)
    for a_w, (dr_w, dc_w) in enumerate(deltas_w):
        if (r_w, c_w) == goal_w:
            ns_w, rew_w = s_w, 0.0
        else:
            nr_w = min(max(r_w + dr_w, 0), H_w - 1)
            nc_w = min(max(c_w + dc_w, 0), W_w - 1)
            ns_w, rew_w = idx_w(nr_w, nc_w), -1.0
        P_w[s_w, a_w, ns_w] = 1.0
        R_w[s_w, a_w, ns_w] = rew_w
print("from state 0:", [(actions_w[a], int(np.argmax(P_w[0, a])), R_w[0, a].sum()) for a in range(A_w)])
assert P_w.shape == (9, 4, 9)

▶ What you'll see: up/left bounce to state 0, while right/down move to states 1 and 3.

In [ ]:
layout_w = np.arange(H_w * W_w).reshape(H_w, W_w)
plt.figure(figsize=(3.6, 3.2))
plt.imshow(layout_w, cmap="Blues")
for rr_w in range(H_w):
    for cc_w in range(W_w):
        label_w = "G" if (rr_w, cc_w) == goal_w else str(idx_w(rr_w, cc_w))
        plt.text(cc_w, rr_w, label_w, ha="center", va="center", fontsize=13)
plt.xticks([]); plt.yticks([]); plt.title("1: gridworld state indices"); plt.show()

▶ What you'll see: a labeled 3×3 grid, with the terminal goal at the lower-right.

*Why it's done this way:* DP needs the transition tensor because a Bellman backup is an expectation over possible next states. The shape `|S|×|A|×|S|` keeps the math honest: values live over states, but backups must first consider each action and next state.

### 2. Discounted return

A reward is local; a return is the discounted future. With rewards `[1, 0, 2]` and discount `0.9`, the delayed reward still matters, but it counts as `0.9²·2` instead of `2`.

In [ ]:
rewards_w = np.array([1.0, 0.0, 2.0])
gamma_w = 0.9
discounts_w = gamma_w ** np.arange(len(rewards_w))
terms_w = discounts_w * rewards_w
print("discounts:", np.round(discounts_w, 3))
print("terms:", np.round(terms_w, 3))

▶ What you'll see: discount powers `[1.00, 0.90, 0.81]`, so the final reward contributes `1.62`.

In [ ]:
G_w = float(np.sum(terms_w))
print("G =", round(G_w, 3))
assert round(G_w, 3) == 2.620
plt.figure(figsize=(4, 3))
plt.bar(["t0", "t1", "t2"], terms_w, color="teal")
plt.title("2: discounted return terms"); plt.ylabel("γ^t r_t"); plt.show()

▶ What you'll see: the total return is 2.620, not the undiscounted sum 3.000.

*Why it's done this way:* optimizing immediate reward alone misses actions that sacrifice now to improve later. Discounting turns a future stream into one finite scalar objective, and `γ < 1` is the condition that lets repeated infinite-horizon backups settle.

### 3. Bellman expectation backup

For a fixed policy, the value of a state is the policy-weighted average of one-step reward plus discounted next value:

$$V^\pi(s)=\sum_a\pi(a\mid s)\sum_{s'}P(s'\mid s,a)\left(R(s,a,s')+\gamma V^\pi(s')\right).$$

The inner sum averages over next states; the outer sum averages over actions chosen by the policy.

In [ ]:
V0_w = np.zeros(H_w * W_w)
pi_uniform_w = np.ones((H_w * W_w, A_w)) / A_w
q0_w = np.array([np.sum(P_w[0, a] * (R_w[0, a] + gamma_w * V0_w)) for a in range(A_w)])
print("Q from state 0 with V=0:", q0_w)

▶ What you'll see: all four one-step action values are −1.

In [ ]:
backup0_w = float(np.dot(pi_uniform_w[0], q0_w))
print("uniform backup for state 0:", round(backup0_w, 3))
assert round(backup0_w, 3) == -1.000

▶ What you'll see: averaging four identical action values gives state value −1.

In [ ]:
V1_w = np.zeros(H_w * W_w)
for s_w in range(H_w * W_w):
    q_s_w = np.array([np.sum(P_w[s_w, a] * (R_w[s_w, a] + gamma_w * V0_w)) for a in range(A_w)])
    V1_w[s_w] = np.dot(pi_uniform_w[s_w], q_s_w)
plt.figure(figsize=(3.8, 3.2))
plt.imshow(V1_w.reshape(H_w, W_w), cmap="viridis")
plt.colorbar(label="value"); plt.title("3: one Bellman sweep"); plt.show()

▶ What you'll see: every non-terminal state becomes −1, while the terminal goal remains 0.

*Why it's done this way:* the Bellman target bootstraps from the current value estimate instead of waiting to enumerate full futures. That is efficient, but the target moves as `V` moves, so one pass is not enough.

### 4. Policy evaluation and convergence

Policy evaluation repeats the expectation backup until the value table stops changing. The max absolute update, `delta`, is the practical convergence signal. With `γ < 1`, the Bellman expectation operator is a contraction, so repeated sweeps approach one fixed value table for the policy.

In [ ]:
V_eval_w = np.zeros(H_w * W_w)
deltas_eval_w = []
for it_w in range(45):
    V_new_w = np.zeros_like(V_eval_w)
    for s_w in range(H_w * W_w):
        for a_w in range(A_w):
            V_new_w[s_w] += pi_uniform_w[s_w, a_w] * np.sum(P_w[s_w, a_w] * (R_w[s_w, a_w] + gamma_w * V_eval_w))
    deltas_eval_w.append(float(np.max(np.abs(V_new_w - V_eval_w))))
    V_eval_w = V_new_w
print("first deltas:", np.round(deltas_eval_w[:5], 3))
print("last delta:", round(deltas_eval_w[-1], 6))

▶ What you'll see: the largest update shrinks as values approach the policy's fixed point.

In [ ]:
print("V for uniform policy:\n", np.round(V_eval_w.reshape(H_w, W_w), 2))
assert V_eval_w[8] == 0.0
plt.figure(figsize=(3.8, 3.2))
plt.imshow(V_eval_w.reshape(H_w, W_w), cmap="magma")
plt.colorbar(label="Vπ(s)"); plt.title("4: uniform-policy values"); plt.show()

▶ What you'll see: states far from the goal have very negative values because random wandering wastes many −1 steps.

In [ ]:
plt.figure(figsize=(4.2, 3))
plt.plot(deltas_eval_w, marker="o", color="purple")
plt.yscale("log"); plt.title("4: policy-evaluation convergence")
plt.xlabel("sweep"); plt.ylabel("max |ΔV|"); plt.show()

▶ What you'll see: a downward curve on a log scale, showing the backup is stabilizing.

*Why it's done this way:* evaluation answers, “If I keep using this policy, what return should I expect?” Each sweep lets one-step consequences propagate farther backward through the grid, and convergence means another sweep would barely change the answer.

### 5. Policy improvement and policy iteration

After evaluating a policy, we improve it by choosing the action with the best one-step lookahead in each state. Policy iteration alternates these two steps: evaluate the current policy, then greedify it. When greedification no longer changes the policy, the policy is stable.

In [ ]:
def q_from_v_w(V_in_w):
    Q_out_w = np.zeros((H_w * W_w, A_w))
    for s_w in range(H_w * W_w):
        for a_w in range(A_w):
            Q_out_w[s_w, a_w] = np.sum(P_w[s_w, a_w] * (R_w[s_w, a_w] + gamma_w * V_in_w))
    return Q_out_w
Q_eval_w = q_from_v_w(V_eval_w)
print("Q at state 0:", np.round(Q_eval_w[0], 2))

▶ What you'll see: right and down are less bad than bouncing up or left from the top-left.

In [ ]:
policy_pi_w = np.ones((H_w * W_w, A_w)) / A_w
changes_w = []
for outer_w in range(8):
    V_pi_w = np.zeros(H_w * W_w)
    for _ in range(80):
        Q_tmp_w = q_from_v_w(V_pi_w)
        V_pi_w = np.sum(policy_pi_w * Q_tmp_w, axis=1)
    old_actions_w = np.argmax(policy_pi_w, axis=1)
    Q_pi_w = q_from_v_w(V_pi_w)
    new_actions_w = np.argmax(Q_pi_w, axis=1)
    policy_pi_w = np.eye(A_w)[new_actions_w]
    changes_w.append(int(np.sum(old_actions_w != new_actions_w)))
    print("iteration", outer_w, "policy changes:", changes_w[-1])
    if changes_w[-1] == 0:
        break

▶ What you'll see: the number of changed states drops to zero after a few evaluate/improve cycles.

In [ ]:
print(np.array([actions_w[a] for a in np.argmax(policy_pi_w, axis=1)]).reshape(H_w, W_w))
plt.figure(figsize=(3.8, 3.2))
plt.imshow(V_pi_w.reshape(H_w, W_w), cmap="viridis")
for rr_w in range(H_w):
    for cc_w in range(W_w):
        s_show_w = idx_w(rr_w, cc_w)
        txt_w = "G" if (rr_w, cc_w) == goal_w else actions_w[np.argmax(policy_pi_w[s_show_w])]
        plt.text(cc_w, rr_w, txt_w, ha="center", va="center", color="white", fontsize=14)
plt.colorbar(label="V(s)"); plt.title("5: policy iteration result"); plt.show()

▶ What you'll see: arrows form shortest paths toward the goal, and values are highest near the terminal cell.

*Why it's done this way:* evaluation measures the consequences of current behavior; improvement changes behavior using those consequences. The policy improvement theorem guarantees that a greedy policy with respect to an evaluated `V` is no worse than the old policy.

### 6. Value iteration

Value iteration combines evaluation and improvement in one optimality backup:

$$V_{k+1}(s)=\max_a\sum_{s'}P(s'\mid s,a)\left(R(s,a,s')+\gamma V_k(s')\right).$$

Instead of fully evaluating each intermediate policy, it takes the max during every sweep and extracts a greedy policy after values converge.

In [ ]:
V_vi_w = np.zeros(H_w * W_w)
deltas_vi_w = []
for it_w in range(30):
    Q_vi_w = q_from_v_w(V_vi_w)
    V_next_w = np.max(Q_vi_w, axis=1)
    deltas_vi_w.append(float(np.max(np.abs(V_next_w - V_vi_w))))
    V_vi_w = V_next_w
print("value-iteration deltas:", np.round(deltas_vi_w[:8], 3))
print("V*:\n", np.round(V_vi_w.reshape(H_w, W_w), 3))

▶ What you'll see: updates vanish quickly, and the optimal values equal negative discounted shortest-path costs.

In [ ]:
Q_star_w = q_from_v_w(V_vi_w)
policy_star_w = np.argmax(Q_star_w, axis=1)
print("optimal actions:", [actions_w[a] for a in policy_star_w])
assert np.allclose(np.round(V_vi_w[[0, 1, 3]], 3), [-3.439, -2.71, -2.71])

▶ What you'll see: from state 0, value is −3.439 because the shortest path pays four discounted step costs.

In [ ]:
plt.figure(figsize=(3.8, 3.2))
plt.imshow(V_vi_w.reshape(H_w, W_w), cmap="viridis")
for rr_w in range(H_w):
    for cc_w in range(W_w):
        s_show_w = idx_w(rr_w, cc_w)
        txt_w = "G" if (rr_w, cc_w) == goal_w else actions_w[policy_star_w[s_show_w]]
        plt.text(cc_w, rr_w, txt_w, ha="center", va="center", color="white", fontsize=14)
plt.colorbar(label="V*(s)"); plt.title("6: value iteration heatmap + arrows"); plt.show()

▶ What you'll see: values become less negative near the goal, and arrows choose shortest paths.

*Why it's done this way:* value iteration is cheaper when full policy evaluation after every improvement is unnecessary. The max backup directly propagates optimal future consequence, and the same contraction logic gives convergence to a unique optimal fixed point.

## 🛠️ Setup

In [ ]:
import numpy as np  # Import NumPy for arrays, transition tensors, Bellman backups, and numerical checks.
import matplotlib.pyplot as plt  # Import Matplotlib for heatmaps, bars, arrows, and convergence curves.
np.random.seed(0)  # Make all examples reproducible.

ACTIONS = ["↑", "→", "↓", "←"]  # Use one action order everywhere.
DELTAS = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # Up, right, down, left.

def grid_index(r, c, width):  # Convert a row/column coordinate to a flat state id.
    return r * width + c

def grid_rc(s, width):  # Convert a flat state id to a row/column coordinate.
    return divmod(int(s), width)

def make_gridworld(height=3, width=3, goal=(2, 2), step_reward=-1.0):  # Build a deterministic gridworld MDP.
    nS, nA = height * width, 4  # Count states and actions.
    P = np.zeros((nS, nA, nS))  # Transition probabilities P[s,a,s'].
    R = np.zeros((nS, nA, nS))  # Rewards R[s,a,s'].
    for s in range(nS):  # Fill each state.
        r, c = grid_rc(s, width)  # Decode coordinates.
        for a, (dr, dc) in enumerate(DELTAS):  # Fill each action.
            if (r, c) == goal:  # Terminal state self-loops.
                ns, rew = s, 0.0  # No more step cost at the goal.
            else:
                nr = min(max(r + dr, 0), height - 1)  # Clip row at boundary.
                nc = min(max(c + dc, 0), width - 1)  # Clip column at boundary.
                ns, rew = grid_index(nr, nc, width), step_reward  # Deterministic move and reward.
            P[s, a, ns] = 1.0  # Store deterministic probability.
            R[s, a, ns] = rew  # Store transition reward.
    return P, R  # Return the known model.

def q_from_v(P, R, V, gamma):  # Compute one-step action values from state values.
    nS, nA, _ = P.shape  # Read dimensions.
    Q = np.zeros((nS, nA))  # Allocate Q[s,a].
    for s in range(nS):  # Loop over states.
        for a in range(nA):  # Loop over actions.
            Q[s, a] = np.sum(P[s, a] * (R[s, a] + gamma * V))  # Bellman lookahead.
    return Q  # Return action-value table.

def eval_policy(P, R, pi, gamma=0.9, sweeps=80):  # Evaluate a policy by repeated Bellman expectation backups.
    V = np.zeros(P.shape[0])  # Initialize values.
    deltas = []  # Track max table changes.
    for _ in range(sweeps):  # Repeat sweeps.
        Q = q_from_v(P, R, V, gamma)  # Compute lookahead values.
        V_new = np.sum(pi * Q, axis=1)  # Average over policy probabilities.
        deltas.append(float(np.max(np.abs(V_new - V))))  # Record convergence diagnostic.
        V = V_new  # Accept updated values.
    return V, np.array(deltas)  # Return values and deltas.

def greedy_policy_from_v(P, R, V, gamma=0.9):  # Greedify one-step lookahead from V.
    Q = q_from_v(P, R, V, gamma)  # Compute Q table.
    best = np.argmax(Q, axis=1)  # Choose best action per state.
    return np.eye(P.shape[1])[best], best, Q  # Return one-hot policy, action ids, and Q.

def plot_grid_values(V, title, height=3, width=3, goal=(2, 2), actions=None):  # Draw a value heatmap with optional arrows.
    plt.figure(figsize=(4, 3.2))  # Create compact figure.
    plt.imshow(V.reshape(height, width), cmap="viridis")  # Draw values as colors.
    for r in range(height):  # Annotate rows.
        for c in range(width):  # Annotate columns.
            s = grid_index(r, c, width)  # Flat state id.
            text = "G" if (r, c) == goal else (ACTIONS[int(actions[s])] if actions is not None else f"{V[s]:.1f}")  # Value or arrow.
            plt.text(c, r, text, ha="center", va="center", color="white", fontsize=12)  # Add annotation.
    plt.colorbar(label="value")  # Show numeric value scale.
    plt.title(title)  # Title the figure.
    plt.xticks([]); plt.yticks([])  # Hide ticks.
    plt.show()  # Display figure.

## 🟢 Basics (warm-up)

### Basic 1 — Build a known transition table

**Goal.** Create a deterministic transition and reward tensor, because DP can only sweep exact backups when the model is known. We build it in 2 steps.

In [ ]:
P_b1, R_b1 = make_gridworld()  # Build the 3x3 gridworld model.
print("P shape:", P_b1.shape)  # Inspect |S| x |A| x |S|.
print("R shape:", R_b1.shape)  # Inspect matching reward shape.

▶ What you'll see: both tensors have shape `(9, 4, 9)`.

In [ ]:
transitions_b1 = [(ACTIONS[a], int(np.argmax(P_b1[0, a])), float(R_b1[0, a].sum())) for a in range(4)]  # Summarize state 0.
print("state 0 transitions:", transitions_b1)  # Inspect boundary and movement behavior.
assert transitions_b1[1][1] == 1 and transitions_b1[2][1] == 3  # Right/down move to neighbors.

▶ What you'll see: boundary actions stay in place, while right and down enter the grid.

In [ ]:
layout_b1 = np.arange(9).reshape(3, 3)  # Reuse the state ids to draw the model spatially.
plt.figure(figsize=(3.8, 3.2))  # Create a compact grid figure.
plt.imshow(layout_b1, cmap="Blues", alpha=0.55)  # Draw cells so transitions have coordinates.
for r_b1 in range(3):
    for c_b1 in range(3):
        plt.text(c_b1, r_b1, str(layout_b1[r_b1, c_b1]), ha="center", va="center", color="black")  # Label states.
start_r_b1, start_c_b1 = grid_rc(0, 3)  # Visualize transitions out of state 0.
for a_b1, name_b1 in enumerate(ACTIONS):
    ns_b1 = int(np.argmax(P_b1[0, a_b1]))  # Destination under this action.
    nr_b1, nc_b1 = grid_rc(ns_b1, 3)  # Destination coordinates.
    if ns_b1 == 0:
        continue  # Boundary actions are annotated in-place below.
    plt.arrow(start_c_b1, start_r_b1, 0.65 * (nc_b1 - start_c_b1), 0.65 * (nr_b1 - start_r_b1),
              head_width=0.08, length_includes_head=True, color="crimson")  # Draw movement arrow.
    plt.text((start_c_b1 + nc_b1) / 2, (start_r_b1 + nr_b1) / 2, name_b1, color="crimson", fontsize=12)
plt.text(start_c_b1 - 0.34, start_r_b1 - 0.28, "↑/← stay", color="crimson", fontsize=10)  # Boundary self-loops.
plt.title("Basic 1: transitions from state 0"); plt.xticks([]); plt.yticks([]); plt.show()

▶ What you'll see: arrows show right/down transitions from state 0, while up/left are labeled as boundary self-loops.

👀 Takeaway: dynamic programming plans from a complete transition-and-reward table.

### Basic 2 — Visualize flat state ids

**Goal.** Map vector indices back to a grid, because value tables are flat but policies are easier to read spatially. We build it in 2 steps.

In [ ]:
states_b2 = np.arange(9)  # Create flat state ids.
coords_b2 = [grid_rc(s, 3) for s in states_b2]  # Convert each id to row/column coordinates.
print("coordinates:", coords_b2)  # Inspect row-major ordering.

▶ What you'll see: state ids increase left-to-right, then top-to-bottom.

In [ ]:
layout_b2 = states_b2.reshape(3, 3)  # Reshape state ids as a grid.
print(layout_b2)  # Inspect the grid layout.
plt.figure(figsize=(3.5, 3))  # Create a compact grid figure.
plt.imshow(layout_b2, cmap="Blues")  # Draw state ids as color.
for r_b2 in range(3):
    for c_b2 in range(3):
        plt.text(c_b2, r_b2, str(layout_b2[r_b2, c_b2]), ha="center", va="center")  # Label cells.
plt.title("Basic 2: state indexing"); plt.xticks([]); plt.yticks([]); plt.show()  # Display grid.

▶ What you'll see: a labeled 3×3 state map.

👀 Takeaway: keep computational shape and visualization shape connected.

### Basic 3 — Compute discounted return

**Goal.** Sum a short reward stream with discount powers, because values represent future return rather than one immediate reward. We build it in 2 steps.

In [ ]:
rewards_b3 = np.array([1.0, 0.0, 2.0])  # Define the worked reward stream.
gamma_b3 = 0.9  # Choose the discount.
weights_b3 = gamma_b3 ** np.arange(len(rewards_b3))  # Compute γ^t.
print("weights:", np.round(weights_b3, 3))  # Inspect discount powers.

▶ What you'll see: the third reward receives weight 0.81.

In [ ]:
return_b3 = float(np.sum(weights_b3 * rewards_b3))  # Compute G = Σγ^t r_t.
print("return:", round(return_b3, 3))  # Inspect the discounted return.
assert round(return_b3, 3) == 2.620  # Verify the worked number.
plt.figure(figsize=(4, 3)); plt.bar(["r0", "r1", "r2"], weights_b3 * rewards_b3, color="teal")
plt.title("Basic 3: return contributions"); plt.ylabel("discounted reward"); plt.show()

▶ What you'll see: the return is 2.620.

👀 Takeaway: return is a discounted future ledger, not just the next reward.

### Basic 4 — Build one bootstrap target

**Goal.** Compute `r + γV(s')`, because Bellman methods replace the rest of the future with a current estimate. We build it in 2 steps.

In [ ]:
reward_b4 = 1.0  # Immediate reward.
next_value_b4 = 0.8  # Current next-state value estimate.
gamma_b4 = 0.9  # Discount factor.
target_b4 = reward_b4 + gamma_b4 * next_value_b4  # Bellman target.
print("target:", round(target_b4, 3))  # Inspect r + γV(s').
assert round(target_b4, 3) == 1.720  # Verify canonical target.

▶ What you'll see: the bootstrap target is 1.720.

In [ ]:
old_q_b4 = 0.4  # Old estimate.
alpha_b4 = 0.5  # Step size toward target.
new_q_b4 = old_q_b4 + alpha_b4 * (target_b4 - old_q_b4)  # Incremental update.
print("updated estimate:", round(new_q_b4, 3))  # Inspect halfway move.
assert round(new_q_b4, 3) == 1.060  # Verify worked update.

▶ What you'll see: the estimate moves halfway from 0.4 toward 1.720.

In [ ]:
components_b4 = np.array([reward_b4, gamma_b4 * next_value_b4])  # Split the Bellman target into its two terms.
plt.figure(figsize=(4.6, 3))  # Create a compact backup figure.
plt.bar(["reward r", "discounted γV(s')"], components_b4, color=["steelblue", "orange"])  # Show target pieces.
plt.axhline(target_b4, color="crimson", linestyle="--", label=f"target={target_b4:.2f}")  # Full target.
plt.scatter([0.5], [old_q_b4], color="black", zorder=3, label=f"old Q={old_q_b4:.1f}")  # Old estimate.
plt.scatter([0.5], [new_q_b4], color="seagreen", zorder=3, label=f"updated={new_q_b4:.2f}")  # Half step.
plt.ylabel("value"); plt.title("Basic 4: Bellman bootstrap target"); plt.legend(); plt.show()

▶ What you'll see: the Bellman target is the sum of immediate reward and discounted next value, with the update moving partway toward it.

👀 Takeaway: bootstrapping learns from a moving value estimate, so update size matters.

### Basic 5 — Average action values under a policy

**Goal.** Turn four action values into one state value, because `Vπ(s)` is an expectation over the policy's action probabilities. We build it in 2 steps.

In [ ]:
q_actions_b5 = np.array([-1.0, -1.0, -1.0, -1.0])  # One-step action values from state 0.
pi_b5 = np.ones(4) / 4  # Uniform action probabilities.
weighted_b5 = pi_b5 * q_actions_b5  # Contribution from each action.
print("weighted contributions:", weighted_b5)  # Inspect expectation pieces.

▶ What you'll see: each action contributes −0.25.

In [ ]:
value_b5 = float(np.sum(weighted_b5))  # Sum action contributions.
print("Vπ:", round(value_b5, 3))  # Inspect state value.
assert value_b5 == -1.0  # Uniform average of four -1 actions.
plt.figure(figsize=(4, 3)); plt.bar(ACTIONS, weighted_b5, color="orange")
plt.title("Basic 5: policy-weighted backup"); plt.ylabel("π(a|s)Q(s,a)"); plt.show()

▶ What you'll see: the four contributions sum to −1.

👀 Takeaway: policy evaluation averages actions; optimal control maximizes them.

### Basic 6 — Run one Bellman sweep

**Goal.** Update every state once from zeros, because DP propagates reward information by sweeping a table. We build it in 2 steps.

In [ ]:
P_b6, R_b6 = make_gridworld()  # Build model.
V_b6 = np.zeros(9)  # Start with zero values.
pi_b6 = np.ones((9, 4)) / 4  # Uniform policy.
Q_b6 = q_from_v(P_b6, R_b6, V_b6, gamma=0.9)  # Compute one-step lookahead.
print("state 0 Q:", Q_b6[0])  # Inspect first state's actions.

▶ What you'll see: all four action values from state 0 are −1.

In [ ]:
V_next_b6 = np.sum(pi_b6 * Q_b6, axis=1)  # Policy expectation backup.
print("one-sweep V:\n", V_next_b6.reshape(3, 3))  # Inspect table after one sweep.
assert V_next_b6[8] == 0.0  # Goal remains terminal zero.
plot_grid_values(V_next_b6, "Basic 6: one Bellman sweep")  # Visualize updated values.

▶ What you'll see: non-terminal states become −1 and the goal stays 0.

👀 Takeaway: one sweep only sees one-step consequences; repeated sweeps propagate farther.

### Basic 7 — Track policy-evaluation convergence

**Goal.** Record max value change across sweeps, because a tiny delta means another backup will barely change the table. We build it in 2 steps.

In [ ]:
P_b7, R_b7 = make_gridworld()  # Build model.
pi_b7 = np.ones((9, 4)) / 4  # Evaluate random policy.
V_b7, deltas_b7 = eval_policy(P_b7, R_b7, pi_b7, gamma=0.9, sweeps=25)  # Repeated backups.
print("first deltas:", np.round(deltas_b7[:6], 3))  # Inspect early changes.

▶ What you'll see: deltas begin large and shrink.

In [ ]:
print("final delta:", round(float(deltas_b7[-1]), 6))  # Inspect late change.
assert deltas_b7[-1] < deltas_b7[0]  # Verify progress.
plt.figure(figsize=(4, 3)); plt.plot(deltas_b7, marker="o", color="purple")
plt.yscale("log"); plt.title("Basic 7: evaluation delta"); plt.xlabel("sweep"); plt.ylabel("max |ΔV|"); plt.show()

▶ What you'll see: a downward convergence curve.

👀 Takeaway: convergence is monitored by table change, not by whether rewards are still nonzero.

### Basic 8 — Convert V to Q

**Goal.** Compute action values from state values, because policy improvement compares actions using one-step lookahead. We build it in 2 steps.

In [ ]:
P_b8, R_b8 = make_gridworld()  # Build model.
V_b8 = np.array([-3.439, -2.71, -1.9, -2.71, -1.9, -1.0, -1.9, -1.0, 0.0])  # Near-optimal values.
Q_b8 = q_from_v(P_b8, R_b8, V_b8, gamma=0.9)  # Compute Q(s,a).
print("Q at state 0:", np.round(Q_b8[0], 3))  # Inspect actions.

▶ What you'll see: right and down are tied as best shortest-path actions.

In [ ]:
best_b8 = int(np.argmax(Q_b8[0]))  # Choose greedy action.
print("best action:", ACTIONS[best_b8])  # Inspect action label.
assert ACTIONS[best_b8] in ["→", "↓"]  # Shortest-path action.
plt.figure(figsize=(4, 3)); plt.bar(ACTIONS, Q_b8[0], color="seagreen")
plt.title("Basic 8: one-step Q from V"); plt.ylabel("Q(0,a)"); plt.show()

▶ What you'll see: moving toward the goal has higher value than bouncing into a wall.

👀 Takeaway: `Q` exposes the action dimension that `V` hides.

### Basic 9 — Greedify a policy

**Goal.** Turn value estimates into deterministic action probabilities, because policy improvement assigns probability 1 to the best action. We build it in 2 steps.

In [ ]:
P_b9, R_b9 = make_gridworld()  # Build model.
V_b9 = np.array([-3.439, -2.71, -1.9, -2.71, -1.9, -1.0, -1.9, -1.0, 0.0])  # Near-optimal values.
pi_b9, acts_b9, Q_b9 = greedy_policy_from_v(P_b9, R_b9, V_b9, gamma=0.9)  # Greedify.
print("greedy action ids:", acts_b9)  # Inspect selected actions.

▶ What you'll see: one action id per state.

In [ ]:
print("policy row 0:", pi_b9[0])  # Inspect one-hot probabilities.
assert np.isclose(pi_b9[0].sum(), 1.0)  # Valid probability row.
plot_grid_values(V_b9, "Basic 9: greedy arrows", actions=acts_b9)  # Draw policy arrows.

▶ What you'll see: arrows point along shortest paths toward the goal.

👀 Takeaway: greedification converts evaluated consequence into behavior.

### Basic 10 — Keep V and Q shapes separate

**Goal.** Inspect table shapes, because a scalar state value and a vector of action values are different mathematical objects. We build it in 2 steps.

In [ ]:
P_b10, R_b10 = make_gridworld()  # Build model.
V_b10 = np.zeros(9)  # One scalar per state.
Q_b10 = q_from_v(P_b10, R_b10, V_b10, gamma=0.9)  # One scalar per state-action pair.
print("V shape:", V_b10.shape)  # Inspect state-value shape.
print("Q shape:", Q_b10.shape)  # Inspect action-value shape.

▶ What you'll see: `V` is `(9,)` while `Q` is `(9, 4)`.

In [ ]:
print("V[4]:", V_b10[4])  # One number for center state.
print("Q[4]:", Q_b10[4])  # Four numbers for center actions.
assert V_b10.shape == (9,) and Q_b10.shape == (9, 4)  # Verify shapes.

▶ What you'll see: a state has one value but four action values.

In [ ]:
fig_b10, axes_b10 = plt.subplots(1, 2, figsize=(7, 3))  # Compare state values and action values side by side.
im_b10 = axes_b10[0].imshow(V_b10.reshape(3, 3), cmap="viridis", vmin=-1.1, vmax=0.1)  # One scalar per state.
for r_b10 in range(3):
    for c_b10 in range(3):
        s_b10 = grid_index(r_b10, c_b10, 3)
        axes_b10[0].text(c_b10, r_b10, f"V{s_b10}\n{V_b10[s_b10]:.1f}", ha="center", va="center", color="white")
axes_b10[0].set_title("V: one number/state"); axes_b10[0].set_xticks([]); axes_b10[0].set_yticks([])
axes_b10[1].bar(ACTIONS, Q_b10[4], color="seagreen")  # Four action values for one chosen state.
axes_b10[1].set_title("Q[4]: four actions"); axes_b10[1].set_ylabel("Q(4,a)"); axes_b10[1].set_ylim(-1.1, 0.1)
fig_b10.colorbar(im_b10, ax=axes_b10[0], fraction=0.046, pad=0.04, label="V(s)")
plt.tight_layout(); plt.show()

▶ What you'll see: the left panel stores one scalar per grid cell, while the right panel expands state 4 into four action values.

👀 Takeaway: shape discipline prevents mixing policy evaluation with action comparison.

## 🟡 Easy

### Easy 1 — Evaluate a fixed random policy

**Goal.** Estimate the value of uniform random behavior, because policy iteration needs current-policy values before improving that policy. We build it in 3 steps.

In [ ]:
P_e1, R_e1 = make_gridworld()  # Build known model.
pi_e1 = np.ones((9, 4)) / 4  # Uniform random policy.
print("policy row 0:", pi_e1[0])  # Inspect action probabilities.

▶ What you'll see: each action has probability 0.25.

In [ ]:
V_e1, deltas_e1 = eval_policy(P_e1, R_e1, pi_e1, gamma=0.9, sweeps=80)  # Evaluate policy.
print("start/end delta:", round(float(deltas_e1[0]), 3), round(float(deltas_e1[-1]), 6))  # Inspect convergence.
print("Vπ:\n", np.round(V_e1.reshape(3, 3), 2))  # Inspect values.

In [ ]:
assert V_e1[8] == 0.0 and V_e1[0] < V_e1[4] < V_e1[8]  # Values improve near the goal.
plot_grid_values(V_e1, "Easy 1: uniform-policy values")  # Visualize state values.

▶ What you'll see: the top-left is very negative under random wandering.

👀 Takeaway: policy evaluation measures the consequence of a fixed behavior rule.

### Easy 2 — Improve a policy after evaluation

**Goal.** Greedify evaluated values, because policy improvement replaces weak random choices with better one-step lookahead choices. We build it in 3 steps.

In [ ]:
P_e2, R_e2 = make_gridworld()  # Build model.
pi_uniform_e2 = np.ones((9, 4)) / 4  # Start from random policy.
V_e2, _ = eval_policy(P_e2, R_e2, pi_uniform_e2, gamma=0.9, sweeps=80)  # Evaluate it.
print("center value:", round(float(V_e2[4]), 3))  # Inspect a representative value.

▶ What you'll see: the center still has negative value under random behavior.

In [ ]:
pi_greedy_e2, acts_e2, Q_e2 = greedy_policy_from_v(P_e2, R_e2, V_e2, gamma=0.9)  # Improve policy.
print("center Q:", np.round(Q_e2[4], 2))  # Inspect action choices at center.
print("center greedy action:", ACTIONS[int(acts_e2[4])])  # Inspect chosen action.

In [ ]:
assert np.isclose(pi_greedy_e2[4].sum(), 1.0)  # Greedy row is valid.
plot_grid_values(V_e2, "Easy 2: improved arrows", actions=acts_e2)  # Show arrows over values.

▶ What you'll see: arrows point toward actions with better continuation values.

👀 Takeaway: improvement uses `Q(s,a)` lookahead even when evaluation stores only `V(s)`.

### Easy 3 — Run policy iteration

**Goal.** Alternate evaluation and improvement until the policy is stable, because a stable greedy policy is optimal in this finite known MDP. We build it in 4 steps.

In [ ]:
P_e3, R_e3 = make_gridworld()  # Build known MDP.
pi_e3 = np.ones((9, 4)) / 4  # Initialize random policy.
changes_e3 = []  # Track changed states.
print("initial argmax actions:", np.argmax(pi_e3, axis=1))  # Inspect initial ties.

▶ What you'll see: the initial argmax is arbitrary because all probabilities tie.

In [ ]:
for outer_e3 in range(8):
    V_e3, _ = eval_policy(P_e3, R_e3, pi_e3, gamma=0.9, sweeps=80)  # Evaluate current policy.
    old_e3 = np.argmax(pi_e3, axis=1)  # Current action ids.
    pi_new_e3, acts_e3, Q_e3 = greedy_policy_from_v(P_e3, R_e3, V_e3, gamma=0.9)  # Improve.
    changes_e3.append(int(np.sum(old_e3 != acts_e3)))  # Count changes.
    pi_e3 = pi_new_e3  # Adopt improved policy.
    print("iteration", outer_e3, "changes", changes_e3[-1])  # Inspect progress.
    if changes_e3[-1] == 0:
        break

In [ ]:
assert changes_e3[-1] == 0  # Stable policy reached.
print("final V:\n", np.round(V_e3.reshape(3, 3), 3))  # Inspect final values.

In [ ]:
plt.figure(figsize=(4, 3)); plt.plot(changes_e3, marker="o", color="teal")
plt.title("Easy 3: policy changes"); plt.xlabel("cycle"); plt.ylabel("changed states"); plt.show()
plot_grid_values(V_e3, "Easy 3: final policy", actions=acts_e3)  # Visualize final policy.

▶ What you'll see: changes reach zero, and arrows form shortest paths to the goal.

👀 Takeaway: policy iteration searches over policies by repeatedly evaluating and greedifying.

### Easy 4 — Run value iteration

**Goal.** Apply Bellman optimality backups until values converge, because value iteration solves control without separately evaluating every intermediate policy. We build it in 4 steps.

In [ ]:
P_e4, R_e4 = make_gridworld()  # Build model.
V_e4 = np.zeros(9)  # Initialize V* estimate.
gamma_e4 = 0.9  # Discount.
deltas_e4 = []  # Track convergence.
print("initial V:\n", V_e4.reshape(3, 3))  # Inspect starting table.

▶ What you'll see: all values begin at zero.

In [ ]:
for sweep_e4 in range(30):
    Q_e4 = q_from_v(P_e4, R_e4, V_e4, gamma_e4)  # Compute lookahead.
    V_new_e4 = np.max(Q_e4, axis=1)  # Bellman optimality backup.
    deltas_e4.append(float(np.max(np.abs(V_new_e4 - V_e4))))  # Record max change.
    V_e4 = V_new_e4  # Update values.
print("deltas:", np.round(deltas_e4[:8], 3))  # Inspect convergence.

In [ ]:
pi_e4, acts_e4, Q_final_e4 = greedy_policy_from_v(P_e4, R_e4, V_e4, gamma=gamma_e4)  # Extract policy.
print("V*:\n", np.round(V_e4.reshape(3, 3), 3))  # Inspect optimal values.
assert np.allclose(np.round(V_e4[[0, 1, 4]], 3), [-3.439, -2.71, -1.9])  # Verify shortest-path values.

In [ ]:
plt.figure(figsize=(4, 3)); plt.plot(deltas_e4, marker="o", color="purple")
plt.title("Easy 4: value-iteration convergence"); plt.xlabel("sweep"); plt.ylabel("max |ΔV|"); plt.show()
plot_grid_values(V_e4, "Easy 4: V* and greedy policy", actions=acts_e4)  # Visualize values and policy.

▶ What you'll see: values converge quickly and the heatmap is highest near the goal.

👀 Takeaway: value iteration bakes greedy improvement into every value update.

### Easy 5 — Compare policy iteration and value iteration

**Goal.** Verify both DP algorithms reach the same optimal fixed point, because their update schedules differ but their Bellman target is the same. We build it in 3 steps.

In [ ]:
P_e5, R_e5 = make_gridworld()  # Build model.
pi_e5 = np.ones((9, 4)) / 4  # Start policy iteration from random policy.
for _ in range(6):
    V_pi_e5, _ = eval_policy(P_e5, R_e5, pi_e5, gamma=0.9, sweeps=80)  # Evaluate.
    pi_next_e5, acts_pi_e5, _ = greedy_policy_from_v(P_e5, R_e5, V_pi_e5, gamma=0.9)  # Improve.
    if np.array_equal(np.argmax(pi_e5, axis=1), acts_pi_e5):
        break
    pi_e5 = pi_next_e5
print("policy iteration complete")  # Confirm loop ended.

▶ What you'll see: policy iteration stabilizes on this small grid.

In [ ]:
V_vi_e5 = np.zeros(9)  # Start value iteration.
for _ in range(30):
    V_vi_e5 = np.max(q_from_v(P_e5, R_e5, V_vi_e5, gamma=0.9), axis=1)  # Optimality backup.
diff_e5 = float(np.max(np.abs(V_pi_e5 - V_vi_e5)))  # Compare value tables.
print("max |PI - VI|:", round(diff_e5, 6))  # Inspect difference.
assert diff_e5 < 1e-3  # Same fixed point.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["max difference"], [diff_e5], color="seagreen")
plt.title("Easy 5: same optimal values"); plt.ylabel("|difference|"); plt.show()

▶ What you'll see: the value difference is essentially zero.

👀 Takeaway: policy iteration and value iteration are different routes to the same optimal Bellman solution.

## 🔴 Advanced

### Advanced 1 — Add stochastic transitions

**Goal.** Solve a slippery gridworld, because Bellman backups average over next-state probabilities when actions are uncertain. We build it in 4 steps.

In [ ]:
P_base_a1, R_base_a1 = make_gridworld()  # Deterministic baseline.
slip_a1 = 0.2  # Probability of slipping to side actions.
P_a1 = np.zeros_like(P_base_a1)  # Stochastic transition table.
R_a1 = np.zeros_like(R_base_a1)  # Reward table.
print("slip probability:", slip_a1)  # Inspect uncertainty level.

▶ What you'll see: intended action probability will be 0.8, with 0.1 on each side slip.

In [ ]:
for s_a1 in range(9):
    for a_a1 in range(4):
        probs_a1 = {a_a1: 1 - slip_a1, (a_a1 - 1) % 4: slip_a1 / 2, (a_a1 + 1) % 4: slip_a1 / 2}
        for real_a1, prob_a1 in probs_a1.items():
            ns_a1 = int(np.argmax(P_base_a1[s_a1, real_a1]))
            rew_a1 = float(R_base_a1[s_a1, real_a1, ns_a1])
            P_a1[s_a1, a_a1, ns_a1] += prob_a1
            R_a1[s_a1, a_a1, ns_a1] = rew_a1
print("state 0, action right probabilities:", np.round(P_a1[0, 1], 2))
assert np.isclose(P_a1[0, 1].sum(), 1.0)

In [ ]:
V_a1 = np.zeros(9)  # Initialize values.
for _ in range(60):
    V_a1 = np.max(q_from_v(P_a1, R_a1, V_a1, gamma=0.9), axis=1)  # Stochastic optimality backups.
_, acts_a1, _ = greedy_policy_from_v(P_a1, R_a1, V_a1, gamma=0.9)  # Extract policy.
print("stochastic V*:\n", np.round(V_a1.reshape(3, 3), 2))  # Inspect values.

In [ ]:
plot_grid_values(V_a1, "Advanced 1: stochastic value iteration", actions=acts_a1)  # Visualize result.

▶ What you'll see: values are more negative than deterministic shortest-path values because slips can waste steps.

👀 Takeaway: transition probabilities enter Bellman backups as exact expectation weights.

### Advanced 2 — Sweep discount factors

**Goal.** Compare optimal start values across `γ`, because discounting defines how much future step cost matters. We build it in 4 steps.

In [ ]:
P_a2, R_a2 = make_gridworld()  # Build deterministic model.
gammas_a2 = np.array([0.2, 0.5, 0.9, 0.99])  # Try short- and long-horizon discounts.
start_values_a2 = []  # Store V*(start).
print("gammas:", gammas_a2)  # Inspect sweep.

▶ What you'll see: the sweep ranges from shortsighted to near-undiscounted.

In [ ]:
for gamma_a2 in gammas_a2:
    V_a2 = np.zeros(9)
    for _ in range(100):
        V_a2 = np.max(q_from_v(P_a2, R_a2, V_a2, gamma=gamma_a2), axis=1)
    start_values_a2.append(V_a2[0])
print("V*(start):", np.round(start_values_a2, 3))

In [ ]:
assert start_values_a2[0] > start_values_a2[-1]  # Larger gamma counts more future costs.
plt.figure(figsize=(5, 3)); plt.plot(gammas_a2, start_values_a2, marker="o", color="navy")
plt.title("Advanced 2: discount changes value"); plt.xlabel("γ"); plt.ylabel("V*(start)"); plt.show()

▶ What you'll see: the start value becomes more negative as γ increases.

In [ ]:
closed_form_a2 = -1 - 0.9 - 0.9**2 - 0.9**3  # Four discounted step costs from top-left to goal.
print("γ=0.9 shortest-path value:", round(closed_form_a2, 3))
assert round(closed_form_a2, 3) == -3.439

▶ What you'll see: the analytic shortest-path return is −3.439.

👀 Takeaway: the discount is part of the objective, not just a convergence trick.

### Advanced 3 — Stop by tolerance

**Goal.** Stop value iteration when max change is tiny, because practical DP uses a tolerance rather than a fixed sweep count. We build it in 4 steps.

In [ ]:
P_a3, R_a3 = make_gridworld()  # Build model.
V_a3 = np.zeros(9)  # Initialize values.
tol_a3 = 1e-4  # Stopping tolerance.
deltas_a3 = []  # Track max changes.
print("tolerance:", tol_a3)  # Inspect stopping rule.

▶ What you'll see: the loop will stop once max update is below 0.0001.

In [ ]:
for sweep_a3 in range(200):
    V_new_a3 = np.max(q_from_v(P_a3, R_a3, V_a3, gamma=0.9), axis=1)
    delta_a3 = float(np.max(np.abs(V_new_a3 - V_a3)))
    deltas_a3.append(delta_a3)
    V_a3 = V_new_a3
    if delta_a3 < tol_a3:
        break
print("sweeps:", len(deltas_a3), "final delta:", deltas_a3[-1])

In [ ]:
assert deltas_a3[-1] < tol_a3
assert np.allclose(np.round(V_a3[[0, 1, 4]], 3), [-3.439, -2.71, -1.9])
plt.figure(figsize=(4, 3)); plt.plot(deltas_a3, marker="o", color="crimson")
plt.axhline(tol_a3, linestyle="--", color="black", label="tolerance")
plt.yscale("log"); plt.title("Advanced 3: tolerance stop"); plt.xlabel("sweep"); plt.ylabel("max |ΔV|"); plt.legend(); plt.show()

▶ What you'll see: the delta curve crosses the tolerance line.

In [ ]:
_, acts_a3, _ = greedy_policy_from_v(P_a3, R_a3, V_a3, gamma=0.9)
plot_grid_values(V_a3, "Advanced 3: converged V*", actions=acts_a3)

▶ What you'll see: tolerance stopping still recovers shortest-path arrows.

👀 Takeaway: stopping by value-table change preserves the Bellman solution while avoiding wasted sweeps.

### Advanced 4 — Show bootstrapping overshoot

**Goal.** Compare standard backups with aggressive relaxed updates, because bootstrapping from a moving target can oscillate or spike if we overshoot. We build it in 4 steps.

In [ ]:
P_a4, R_a4 = make_gridworld()  # Build model.
pi_a4 = np.ones((9, 4)) / 4  # Evaluate random policy.
relaxations_a4 = [1.0, 1.6]  # Standard replacement versus aggressive over-relaxation.
curves_a4 = []  # Store update-size curves.
print("relaxation factors:", relaxations_a4)

▶ What you'll see: `1.0` replaces with the Bellman target; `1.6` steps beyond it.

In [ ]:
for omega_a4 in relaxations_a4:
    V_a4 = np.zeros(9)
    curve_a4 = []
    for _ in range(30):
        target_a4 = np.sum(pi_a4 * q_from_v(P_a4, R_a4, V_a4, gamma=0.9), axis=1)
        V_new_a4 = V_a4 + omega_a4 * (target_a4 - V_a4)
        curve_a4.append(float(np.max(np.abs(V_new_a4 - V_a4))))
        V_a4 = V_new_a4
    curves_a4.append(curve_a4)
print("final deltas:", [round(c[-1], 4) for c in curves_a4])

In [ ]:
normal_a4 = np.array(curves_a4[0])
aggressive_a4 = np.array(curves_a4[1])
print("normal first/last:", round(normal_a4[0], 3), round(normal_a4[-1], 3))
print("aggressive max delta:", round(float(np.max(aggressive_a4)), 3))
assert normal_a4[-1] < normal_a4[0]

In [ ]:
plt.figure(figsize=(5, 3)); plt.plot(normal_a4, label="ω=1.0 standard", color="teal")
plt.plot(aggressive_a4, label="ω=1.6 aggressive", color="red")
plt.yscale("log"); plt.title("Advanced 4: bootstrap update size"); plt.xlabel("sweep"); plt.ylabel("max update"); plt.legend(); plt.show()

▶ What you'll see: standard backups settle smoothly; aggressive updates can spike because they move beyond a target that is itself changing.

👀 Takeaway: bootstrapping is powerful, but update rules still need stability discipline.

### Advanced 5 — Compare synchronous and in-place sweeps

**Goal.** Contrast synchronous value iteration with in-place updates, because sweep schedule changes speed even when the fixed point is the same. We build it in 4 steps.

In [ ]:
P_a5, R_a5 = make_gridworld()  # Build model.
tol_a5 = 1e-4  # Shared stopping tolerance.
print("tolerance:", tol_a5)  # Inspect fair stopping rule.

▶ What you'll see: both methods use the same convergence threshold.

In [ ]:
V_sync_a5 = np.zeros(9)
sync_deltas_a5 = []
for _ in range(100):
    V_new_a5 = np.max(q_from_v(P_a5, R_a5, V_sync_a5, gamma=0.9), axis=1)
    delta_a5 = float(np.max(np.abs(V_new_a5 - V_sync_a5)))
    sync_deltas_a5.append(delta_a5)
    V_sync_a5 = V_new_a5
    if delta_a5 < tol_a5:
        break
print("sync sweeps:", len(sync_deltas_a5))

In [ ]:
V_inplace_a5 = np.zeros(9)
inplace_deltas_a5 = []
for _ in range(100):
    old_sweep_a5 = V_inplace_a5.copy()
    for s_a5 in range(9):
        V_inplace_a5[s_a5] = np.max(q_from_v(P_a5, R_a5, V_inplace_a5, gamma=0.9)[s_a5])
    delta_in_a5 = float(np.max(np.abs(V_inplace_a5 - old_sweep_a5)))
    inplace_deltas_a5.append(delta_in_a5)
    if delta_in_a5 < tol_a5:
        break
print("in-place sweeps:", len(inplace_deltas_a5))

In [ ]:
diff_a5 = float(np.max(np.abs(V_sync_a5 - V_inplace_a5)))
print("max final difference:", round(diff_a5, 6))
assert diff_a5 < 1e-3
plt.figure(figsize=(5, 3)); plt.plot(sync_deltas_a5, marker="o", label="synchronous")
plt.plot(inplace_deltas_a5, marker="s", label="in-place")
plt.yscale("log"); plt.title("Advanced 5: sweep schedule"); plt.xlabel("sweep"); plt.ylabel("max |ΔV|"); plt.legend(); plt.show()

▶ What you'll see: both curves reach the same values, often with different sweep counts.

👀 Takeaway: Bellman math defines the fixed point; implementation schedule affects how quickly values propagate.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Dynamic programming plans by sweeping exact backups through a tiny known world.

Dynamic programming assumes the transition model is known. Policy iteration evaluates and improves a policy; value iteration folds improvement into each Bellman sweep. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

SEED = 1106
rng = np.random.default_rng(SEED)
GAMMA = 0.90
ACTIONS = ["up", "right", "down", "left"]
DELTAS = {
    "up": (-1, 0),
    "right": (0, 1),
    "down": (1, 0),
    "left": (0, -1),
}

@dataclass
class GridEnv:
    name: str
    rows: int
    cols: int
    start: int
    states: list
    index_of: dict
    terminal: set
    P: list
    rewards: np.ndarray
    shape_label: str


def discounted_return(rewards, gamma):
    total = 0.0
    power = 1.0
    for reward in rewards:
        total = total + power * reward
        power = power * gamma
    return total


def softmax(logits):
    shifted = np.asarray(logits, dtype=float) - np.max(logits)
    weights = np.exp(shifted)
    return weights / weights.sum()


def move_cell(cell, action, rows, cols, walls):
    dr, dc = DELTAS[action]
    nr = cell[0] + dr
    nc = cell[1] + dc
    candidate = (nr, nc)
    if nr < 0 or nr >= rows:
        return cell
    if nc < 0 or nc >= cols:
        return cell
    if candidate in walls:
        return cell
    return candidate


def build_grid_env(name, rows, cols, start_cell, goal_cells, pit_cells=None, walls=None, step_cost=-0.02, slip=0.0, wind=0.0, bonuses=None):
    pit_cells = set(pit_cells or [])
    walls = set(walls or [])
    bonuses = dict(bonuses or {})
    goal_cells = dict(goal_cells)
    states = []
    for r in range(rows):
        for c in range(cols):
            if (r, c) not in walls:
                states.append((r, c))
    index_of = {cell: i for i, cell in enumerate(states)}
    terminal_cells = set(goal_cells) | pit_cells
    terminal = {index_of[cell] for cell in terminal_cells}
    n_states = len(states)
    rewards = np.zeros(n_states)
    for cell, reward in goal_cells.items():
        rewards[index_of[cell]] = reward
    for cell in pit_cells:
        rewards[index_of[cell]] = -1.0
    for cell, reward in bonuses.items():
        rewards[index_of[cell]] = reward
    P = []
    for state_index, cell in enumerate(states):
        state_rows = []
        for action in ACTIONS:
            if state_index in terminal:
                state_rows.append([(1.0, state_index, 0.0, True)])
                continue
            side_actions = [action, ACTIONS[(ACTIONS.index(action) - 1) % 4], ACTIONS[(ACTIONS.index(action) + 1) % 4]]
            probs = [1.0 - slip, slip / 2.0, slip / 2.0]
            outcomes = {}
            for prob, actual_action in zip(probs, side_actions):
                if prob <= 0.0:
                    continue
                next_cell = move_cell(cell, actual_action, rows, cols, walls)
                windy_cell = move_cell(next_cell, "up", rows, cols, walls)
                wind_options = [(1.0 - wind, next_cell), (wind, windy_cell)]
                for wind_prob, final_cell in wind_options:
                    if wind_prob <= 0.0:
                        continue
                    next_index = index_of[final_cell]
                    done = next_index in terminal
                    reward = step_cost + rewards[next_index]
                    key = (next_index, done, reward)
                    outcomes[key] = outcomes.get(key, 0.0) + prob * wind_prob
            state_rows.append([(prob, ns, rew, done) for (ns, done, rew), prob in outcomes.items()])
        P.append(state_rows)
    shape_label = f"{rows}x{cols}, |S|={n_states}, |A|={len(ACTIONS)}"
    return GridEnv(name, rows, cols, index_of[start_cell], states, index_of, terminal, P, rewards, shape_label)


def two_state_chain():
    return build_grid_env(
        "D1 two-state chain",
        1,
        2,
        (0, 0),
        {(0, 1): 1.0},
        step_cost=0.0,
        slip=0.0,
    )


def build_env_ladder():
    envs = []
    envs.append(two_state_chain())
    envs.append(build_grid_env(
        "D2 slippery 3-state",
        1,
        3,
        (0, 0),
        {(0, 2): 1.0},
        pit_cells={(0, 1)},
        step_cost=-0.01,
        slip=0.20,
    ))
    envs.append(build_grid_env(
        "D3 4x4 gridworld",
        4,
        4,
        (3, 0),
        {(0, 3): 1.0},
        pit_cells={(1, 3)},
        walls={(1, 1), (2, 1)},
        step_cost=-0.03,
        slip=0.05,
    ))
    envs.append(build_grid_env(
        "D4 stochastic windy grid",
        5,
        5,
        (4, 0),
        {(0, 4): 1.2},
        pit_cells={(2, 3), (3, 2)},
        walls={(1, 1), (1, 2), (3, 1)},
        step_cost=-0.04,
        slip=0.15,
        wind=0.20,
    ))
    envs.append(build_grid_env(
        "D5 larger sparse-reward grid",
        8,
        8,
        (7, 0),
        {(0, 7): 2.0},
        pit_cells={(2, 5), (3, 5), (5, 3), (6, 6)},
        walls={(1, 1), (1, 2), (1, 3), (2, 1), (4, 2), (4, 3), (4, 4), (5, 5)},
        step_cost=-0.025,
        slip=0.10,
        wind=0.10,
        bonuses={(7, 1): 0.25},
    ))
    return envs


def q_from_v(env, V, gamma=GAMMA):
    Q = np.zeros((len(env.states), len(ACTIONS)))
    for s in range(len(env.states)):
        for a in range(len(ACTIONS)):
            total = 0.0
            for prob, next_state, reward, done in env.P[s][a]:
                total = total + prob * (reward + gamma * V[next_state] * (not done))
            Q[s, a] = total
    return Q


def policy_evaluation(env, policy, gamma=GAMMA, sweeps=200, tol=1e-10):
    V = np.zeros(len(env.states))
    errors = []
    for sweep in range(sweeps):
        old = V.copy()
        for s in range(len(env.states)):
            if s in env.terminal:
                V[s] = 0.0
                continue
            total = 0.0
            for a in range(len(ACTIONS)):
                for prob, next_state, reward, done in env.P[s][a]:
                    total = total + policy[s, a] * prob * (reward + gamma * old[next_state] * (not done))
            V[s] = total
        errors.append(float(np.max(np.abs(V - old))))
        if errors[-1] < tol:
            break
    return V, np.asarray(errors)


def value_iteration(env, gamma=GAMMA, sweeps=500, tol=1e-10):
    V = np.zeros(len(env.states))
    errors = []
    residuals = []
    for sweep in range(sweeps):
        old = V.copy()
        Q = q_from_v(env, old, gamma)
        for s in range(len(env.states)):
            if s in env.terminal:
                V[s] = 0.0
            else:
                V[s] = np.max(Q[s])
        residual = float(np.max(np.abs(V - old)))
        errors.append(residual)
        residuals.append(residual)
        if residual < tol:
            break
    policy = np.zeros((len(env.states), len(ACTIONS)))
    greedy = np.argmax(q_from_v(env, V, gamma), axis=1)
    for s, action in enumerate(greedy):
        policy[s, action] = 1.0
    return V, policy, np.asarray(errors), np.asarray(residuals)


def policy_iteration(env, gamma=GAMMA, sweeps=80):
    policy = np.ones((len(env.states), len(ACTIONS))) / len(ACTIONS)
    errors = []
    for sweep in range(sweeps):
        V, eval_errors = policy_evaluation(env, policy, gamma=gamma, sweeps=200)
        Q = q_from_v(env, V, gamma)
        greedy = np.argmax(Q, axis=1)
        new_policy = np.zeros_like(policy)
        for s, action in enumerate(greedy):
            new_policy[s, action] = 1.0
        change = float(np.max(np.abs(new_policy - policy)))
        errors.append(change)
        policy = new_policy
        if change == 0.0:
            break
    V, eval_errors = policy_evaluation(env, policy, gamma=gamma, sweeps=300)
    return V, policy, np.asarray(errors)


def run_episode(env, policy, gamma=GAMMA, max_steps=120, epsilon=0.0, start_state=None, rng=None):
    rng = rng or np.random.default_rng(SEED)
    state = env.start if start_state is None else start_state
    trajectory = []
    for step in range(max_steps):
        if state in env.terminal:
            break
        if rng.random() < epsilon:
            action = int(rng.integers(len(ACTIONS)))
        else:
            probs = policy[state] / policy[state].sum()
            action = int(rng.choice(len(ACTIONS), p=probs))
        choices = env.P[state][action]
        probs = np.array([item[0] for item in choices], dtype=float)
        probs = probs / probs.sum()
        choice = int(rng.choice(len(choices), p=probs))
        prob, next_state, reward, done = choices[choice]
        trajectory.append((state, action, reward, next_state, done))
        state = next_state
        if done:
            break
    return trajectory


def episode_return(trajectory, gamma=GAMMA):
    return discounted_return([step[2] for step in trajectory], gamma)


def monte_carlo_value(env, policy, episodes=300, gamma=GAMMA, epsilon=0.10, exploring_starts=False, rng=None):
    rng = rng or np.random.default_rng(SEED)
    V = np.zeros(len(env.states))
    counts = np.zeros(len(env.states))
    errors = []
    V_star, optimal_policy, vi_errors, residuals = value_iteration(env, gamma=gamma)
    for episode in range(episodes):
        start_state = None
        if exploring_starts:
            candidates = [s for s in range(len(env.states)) if s not in env.terminal]
            start_state = int(rng.choice(candidates))
        trajectory = run_episode(env, policy, gamma=gamma, epsilon=epsilon, start_state=start_state, rng=rng)
        G = 0.0
        seen = set()
        for state, action, reward, next_state, done in reversed(trajectory):
            G = reward + gamma * G
            if state in seen:
                continue
            seen.add(state)
            counts[state] = counts[state] + 1.0
            V[state] = V[state] + (G - V[state]) / counts[state]
        errors.append(float(np.max(np.abs(V - V_star))))
    return V, counts, np.asarray(errors)


def td0_value(env, policy, episodes=300, alpha=0.20, gamma=GAMMA, epsilon=0.10, rng=None):
    rng = rng or np.random.default_rng(SEED)
    V = np.zeros(len(env.states))
    errors = []
    V_star, optimal_policy, vi_errors, residuals = value_iteration(env, gamma=gamma)
    for episode in range(episodes):
        trajectory = run_episode(env, policy, gamma=gamma, epsilon=epsilon, rng=rng)
        for state, action, reward, next_state, done in trajectory:
            target = reward + gamma * V[next_state] * (not done)
            V[state] = V[state] + alpha * (target - V[state])
        errors.append(float(np.max(np.abs(V - V_star))))
    return V, np.asarray(errors)


def evaluate_policy_return(env, policy, episodes=80, gamma=GAMMA, rng=None):
    rng = rng or np.random.default_rng(SEED)
    returns = []
    for episode in range(episodes):
        trajectory = run_episode(env, policy, gamma=gamma, rng=rng)
        returns.append(episode_return(trajectory, gamma))
    return float(np.mean(returns))


def immediate_reward_policy(env):
    policy = np.zeros((len(env.states), len(ACTIONS)))
    for s in range(len(env.states)):
        if s in env.terminal:
            policy[s, 0] = 1.0
            continue
        means = []
        for a in range(len(ACTIONS)):
            means.append(sum(prob * reward for prob, next_state, reward, done in env.P[s][a]))
        policy[s, int(np.argmax(means))] = 1.0
    return policy


def uniform_policy(env):
    return np.ones((len(env.states), len(ACTIONS))) / len(ACTIONS)


def value_grid(env, V):
    grid = np.full((env.rows, env.cols), np.nan)
    for idx, cell in enumerate(env.states):
        grid[cell] = V[idx]
    return grid


def policy_grid(env, policy):
    chars = np.full((env.rows, env.cols), " ", dtype=object)
    arrows = np.array(["^", ">", "v", "<"], dtype=object)
    greedy = np.argmax(policy, axis=1)
    for idx, cell in enumerate(env.states):
        chars[cell] = "T" if idx in env.terminal else arrows[greedy[idx]]
    return chars


def plot_value_policy_panels(envs, values, policies, metric_values, metric_name):
    fig, axes = plt.subplots(2, len(envs), figsize=(4 * len(envs), 7))
    for i, env in enumerate(envs):
        ax = axes[0, i]
        image = ax.imshow(value_grid(env, values[i]), cmap="viridis")
        ax.set_title(env.name)
        plt.colorbar(image, ax=ax, fraction=0.046)
        ax = axes[1, i]
        ax.imshow(value_grid(env, values[i]), cmap="viridis")
        arrows = policy_grid(env, policies[i])
        grid = value_grid(env, values[i])
        for r in range(env.rows):
            for c in range(env.cols):
                if not np.isnan(grid[r, c]):
                    ax.text(c, r, arrows[r, c], ha="center", va="center", color="white")
        ax.set_title("greedy policy")
    fig.tight_layout()
    plt.figure(figsize=(7, 3))
    plt.plot(range(1, len(metric_values) + 1), metric_values, marker="o")
    plt.xticks(range(1, len(metric_values) + 1), ["D1", "D2", "D3", "D4", "D5"])
    plt.ylabel(metric_name)
    plt.xlabel("environment rung")
    plt.title(f"{metric_name} across the D1-D5 ladder")
    plt.grid(True, alpha=0.3)
    plt.show()


def print_ladder_preview(envs):
    for env in envs:
        sample = env.states[: min(5, len(env.states))]
        print(f"{env.name}: {env.shape_label}; start={env.states[env.start]}; sample={sample}")


## The concept, built once on D1

We compare policy iteration and value iteration on the same known-model ladder and verify D1 values by hand.

Formula: $V_{k+1}(s)=\max_a\sum_{s'}P(s'\mid s,a)(R(s,a,s')+\gamma V_k(s'))$

First assert the exact worked numbers from the lesson: discounted return, one-step TD target, softmax policy weighting, and UCB exploration pressure. These are small enough to verify by hand.

In [ ]:
lesson_return = discounted_return([1.0, 0.0, 2.0], 0.9)
td_target = 1.0 + 0.9 * 0.8
q_new = 0.4 + 0.5 * (td_target - 0.4)
policy_probs = softmax([1.0, 0.0])
expected_reward = policy_probs[0] * 2.0 + policy_probs[1] * 0.0
ucb_index = 0.55 + np.sqrt(2.0 * np.log(20.0) / 5.0)
assert np.isclose(lesson_return, 2.620)
assert np.isclose(td_target, 1.720)
assert np.isclose(q_new, 1.060)
assert np.isclose(np.round(policy_probs[0], 3), 0.731)
assert np.isclose(np.round(policy_probs[1], 3), 0.269)
assert np.isclose(np.round(expected_reward, 3), 1.462)
assert np.isclose(np.round(ucb_index, 3), 1.645)
print(lesson_return, td_target, q_new, policy_probs, expected_reward, ucb_index)

Value iteration repeatedly applies exact Bellman optimality backups until the value table stops changing.

In [ ]:
def policy_value_iteration(env, gamma=GAMMA):
    V_value, policy_value, value_errors, residuals = value_iteration(env, gamma=gamma)
    V_policy, policy_policy, policy_errors = policy_iteration(env, gamma=gamma)
    return V_value, policy_value, V_policy, policy_policy, value_errors, policy_errors

env = two_state_chain()
V_value, policy_value, V_policy, policy_policy, value_errors, policy_errors = policy_value_iteration(env)
assert np.isclose(V_value[0], 1.0)
assert np.isclose(V_policy[0], 1.0)
print(V_value, V_policy)

Policy iteration stores one action distribution per state, while value iteration stores one value per state.

In [ ]:
assert policy_value.shape == (2, 4)
assert V_value.shape == (2,)
assert int(np.argmax(policy_value[0])) == 1
print(policy_value)

## The dataset ladder

The family F12 ladder is built inline: D1 two-state chain, D2 slippery three-state, D3 4x4 gridworld, D4 stochastic windy grid, and D5 larger sparse-reward grid.

In [ ]:
envs = build_env_ladder()
print_ladder_preview(envs)

## Run the same method across D1-D5

Collect the plan metric: value-error vs known optimum.

In [ ]:
envs = build_env_ladder()
values = []
policies = []
metrics = []
for env in envs:
    V_star, policy_star, errors, residuals = value_iteration(env)
    one_step_policy = immediate_reward_policy(env)
    V_one_step, eval_errors = policy_evaluation(env, one_step_policy)
    value_error = float(np.max(np.abs(V_one_step - V_star)))
    values.append(V_star)
    policies.append(policy_star)
    metrics.append(value_error)
    print(f"{env.name:28s}  {value_error: .3f}")

## Results visualization

The closing figure has value/policy heatmap panels for every environment plus one summary curve over D1-D5.

In [ ]:
plot_value_policy_panels(envs, values, policies, metrics, "value-error vs known optimum")

## Pitfall on the hardest rung

Reproduce the named D5 pitfall, then apply the fix from the lesson.

In [ ]:
env = envs[-1]
one_step_policy = immediate_reward_policy(env)
V_one_step, one_step_errors = policy_evaluation(env, one_step_policy)
V_dp, policy_dp, errors, residuals = value_iteration(env)
one_step_return = evaluate_policy_return(env, one_step_policy, rng=np.random.default_rng(SEED))
dp_return = evaluate_policy_return(env, policy_dp, rng=np.random.default_rng(SEED))
print(f"one-step greedy return: {one_step_return:.3f}")
print(f"full DP return: {dp_return:.3f}")
assert dp_return > one_step_return

## Evaluate it + Practice

- Metric: value-error vs known optimum on D1-D5, compared with a no-skill uniform or immediate-reward baseline.
- Sanity check: D1 must match the hand value and the lesson numbers asserted above.
- Ablation: turn off discounted consequence or coverage and verify the metric worsens.
- Failure signal: residuals stop shrinking, value shapes mismatch, or D5 return drops below the baseline.
- Reproducibility: keep the provided seed and do not download simulators.

Practice prompts:
1. Change $\gamma$ from $0.90$ to $0.70$ and predict which rungs lose the most value before running.

2. Add one wall to D3 and inspect how the optimal policy heatmap reroutes around it.

3. On D5, compare the no-skill uniform policy with the learned or planned policy using the same return metric.